# 1. 实战Kaggle比赛图像分类CIFAR10

# 📚 本章知识点总结：实战 Kaggle CIFAR-10 图像分类

> 本章把前面学到的「图像增广 + ResNet + 训练技巧」整合成一个**完整的端到端竞赛流程**。
> 重点不在模型多复杂，而在于**真实数据从「一堆原始图片」到「提交 csv」的整条工程链路**。

## 一、整体流程（端到端 pipeline）

```
原始数据                组织数据                构建 Dataset/DataLoader        训练               预测提交
─────────  ──────────────────────  ─────────────────────────────  ───────────  ──────────────
train/*.png  ──读 csv 标签──>  按类名建文件夹     ──ImageFolder──>  带增广的迭代器  ──ResNet18──>  生成 submission.csv
trainLabels.csv             拆出 train/valid               + DataLoader                训练 + 验证
test/*.png                  test 放 unknown/
```

## 二、核心知识点

### 1. 数据组织：`ImageFolder` 的约定
- `torchvision.datasets.ImageFolder` 要求**目录结构即标签**：`根目录/类名/图片.png`。
- 因此我们必须把扁平的 `train/` 目录按标签**重新整理**成 `类名/` 子文件夹，这是本章最繁琐但最关键的工程步骤。
- 测试集没有标签，但 `ImageFolder` 仍要求有一层子目录，所以统一放进 `test/unknown/`。

### 2. 训练集 / 验证集拆分
- 从带标签的训练数据中切出一部分作**验证集**（`valid_ratio=0.1`，即 90% 训练、10% 验证），用于在**无法看到测试标签**的情况下评估模型。
- `train_valid`（训练+验证全部数据）用于**最终提交前**的训练，把所有可用数据都喂给模型以榨干性能。

### 3. 图像增广（仅对训练集）
- 训练集做随机增广：`Resize → RandomResizedCrop → RandomHorizontalFlip`，人为扩充数据分布、抑制过拟合。
- 测试/验证集**只做确定性变换**（`ToTensor + Normalize`），保证评估稳定可复现。
- `Normalize` 用的均值/标准差是 CIFAR-10 数据集在 ImageNet 风格统计下的经典常数。

### 4. 模型与训练技巧
- 模型用 `d2l.resnet18(num_classes=10, in_channels=3)`，适配 32×32 彩色小图。
- **学习率调度** `StepLR(lr_period, lr_decay)`：每隔 `lr_period` 个 epoch 把学习率乘以 `lr_decay`，让训练后期更精细地收敛。
- **权重衰减** `weight_decay`（L2 正则）+ **动量** `momentum=0.9`，是图像任务 SGD 的标准配置。
- `nn.DataParallel` 支持单机多卡数据并行训练。

### 5. 预测与提交
- Kaggle 要求 `id` 按**字符串字典序**排序（`'1','10','100','2'...`），所以 `sort(key=lambda x: str(x))`，这是新手最易踩的坑。
- 预测时把模型输出 `argmax` 得到类别索引，再用 `classes` 列表映射回类名字符串。

## ⚠️ 本 notebook 已修复的两个运行错误
1. **训练函数参数名笔误**：原 `def train(..., num_epoch, ...)` 形参写成单数，函数体却用复数 `num_epochs` → 触发 `NameError`，已统一为 `num_epochs`。
2. **预测时模型未在 GPU 上**：原 `train()` 内部 `net = nn.DataParallel(net).to(...)` 只改了**局部变量**，外层 `net` 仍在 CPU，cell-14 预测会报设备不匹配错误 → 已让 `train()` **返回包装后的 net**，并在调用处接收。

---

In [ ]:
import collections           # 提供 Counter，用于统计每个类别的图片数量
import math                   # 用于 floor 向下取整（计算验证集数量）
import os                     # 路径拼接、目录遍历
import shutil                 # 文件复制等文件系统操作（copy/makedirs 的好搭档）
import pandas as pd           # 读写 csv，最后生成提交文件 submission.csv
import torch
import torchvision            # 计算机视觉工具库：数据集、图像变换、经典模型
from torch import nn
from d2l import torch as d2l  # 李沐 d2l 课程封装好的训练/绘图/计时工具

In [ ]:
# ── 下载数据集 ──────────────────────────────────────────────────────────
# 完整 CIFAR-10 有 5 万张训练图，这里用官方提供的「小样本」方便快速跑通流程：
#   cifar10_tiny = 每个类别取前 1000 张训练图 + 每个类别 5 张测试图
# DATA_HUB 是 d2l 维护的「数据名 -> (下载url, 校验哈希)」注册表
d2l.DATA_HUB['cifar10_tiny'] = (d2l.DATA_URL + 'kaggle_cifar10_tiny.zip',
                                '2068874e4b9a9f0fb07ebe0ad2b29754449ccacd')

demo = True   # True=用小样本快速跑通；False=用下载好的完整数据集（路径见 else 分支）

if demo:
    # download_extract: 若本地无缓存则下载 zip 并解压，返回解压后的目录路径
    data_dir = d2l.download_extract('cifar10_tiny')
else:
    data_dir = '../data/cifar-10'   # 使用完整数据集时，手动指向其所在目录

In [ ]:
# ── 读取标签文件 ────────────────────────────────────────────────────────
# trainLabels.csv 的格式为两列：id,label  例如  1,frog / 2,truck ...
def read_csv_labels(fname):
    """读取 fname，返回 {图片名(不含扩展名): 类别标签} 的字典。"""
    with open(fname, 'r') as f:
        # readlines() 按行读入；[1:] 跳过第一行的表头 "id,label"
        lines = f.readlines()[1:]
    # 对每一行去掉行尾换行符再按逗号切分，得到 [['1','frog'], ['2','truck'], ...]
    tokens = [l.rstrip().split(',') for l in lines]
    # 把每行的 (name, label) 收集成字典：{'1':'frog', '2':'truck', ...}
    return dict(((name, label) for name, label in tokens))

labels = read_csv_labels(os.path.join(data_dir, 'trainLabels.csv'))
labels   # 直接显示字典，确认读取是否正确

In [ ]:
# ── 拆分训练集 / 验证集，并按类名整理到子文件夹 ──────────────────────────
# 目标目录结构（ImageFolder 要求「目录名即类名」）：
#   train_valid_test/
#       train_valid/<类名>/...   全部带标签数据（最终提交训练用）
#       train/<类名>/...         拆分后的训练子集
#       valid/<类名>/...         拆分后的验证子集
def copyfile(filename, target_dir):
    """将单个文件复制到 target_dir（目录不存在则自动创建）。"""
    os.makedirs(target_dir, exist_ok=True)   # exist_ok=True：目录已存在也不报错
    shutil.copy(filename, target_dir)

def reorg_train_valid(data_dir, labels, valid_ratio):
    # Counter 统计每个类别有多少张图；most_common()[-1] 取数量最少的类别
    # [-1][1] 即「样本最少的那个类别的图片数 n」——以最稀有类别为基准定验证集大小
    n = collections.Counter(labels.values()).most_common()[-1][1]
    # 每个类别放入验证集的张数（至少 1 张）
    n_valid_per_label = max(1, math.floor(n * valid_ratio))
    label_count = {}   # 记录每个类别已经放进验证集多少张
    for train_file in os.listdir(os.path.join(data_dir, 'train')):
        # 文件名形如 "1.png"，split('.')[0] 取 "1" 作为 key 查标签
        label = labels[train_file.split('.')[0]]
        fname = os.path.join(data_dir, 'train', train_file)
        # ① 每张图都复制一份到 train_valid（全量数据，最终训练用）
        copyfile(fname, os.path.join(data_dir, 'train_valid_test', 'train_valid', label))
        # ② 该类验证集还没放满 -> 放入 valid；否则放入 train
        if label not in label_count or label_count[label] < n_valid_per_label:
            copyfile(fname, os.path.join(data_dir, 'train_valid_test', 'valid', label))
            label_count[label] = label_count.get(label, 0) + 1
        else:
            copyfile(fname, os.path.join(data_dir, 'train_valid_test', 'train', label))
    return n_valid_per_label

In [ ]:
# ── 整理测试集 ──────────────────────────────────────────────────────────
# 测试集没有标签，但 ImageFolder 仍要求「根目录/某子目录/图片」的结构，
# 所以把所有测试图统一放进一个占位类别目录 unknown/ 下。
def reorg_test(data_dir):
    for test_file in os.listdir(os.path.join(data_dir, 'test')):
        copyfile(os.path.join(data_dir, 'test', test_file),
                 os.path.join(data_dir, 'train_valid_test', 'test', 'unknown'))

In [ ]:
# ── 调用上面定义的函数，真正执行数据整理 ────────────────────────────────
# 注意：前面几个 cell 只是「定义函数」，这里才是「调用执行」。
def reorg_cifar10_data(data_dir, valid_ratio):
    labels = read_csv_labels(os.path.join(data_dir, 'trainLabels.csv'))
    reorg_train_valid(data_dir, labels, valid_ratio)   # 整理 train/valid/train_valid
    reorg_test(data_dir)                               # 整理 test/unknown

batch_size = 32 if demo else 128   # 小样本用小批量；完整数据集用更大批量
valid_ratio = 0.1                  # 训练数据中 10% 划作验证集，90% 用于训练
reorg_cifar10_data(data_dir, valid_ratio)

In [ ]:
# ── 图像增广 / 预处理 ───────────────────────────────────────────────────
# 训练集变换：带随机性，用于扩充数据、抑制过拟合
transform_train = torchvision.transforms.Compose([
    torchvision.transforms.Resize(40),                  # 先放大到 40×40，给随机裁剪留余地
    # 随机裁剪出 32×32 区域：scale 控制裁剪面积占比(64%~100%)，ratio=1 表示保持正方形
    torchvision.transforms.RandomResizedCrop(32, scale=(0.64, 1.0), ratio=(1.0, 1.0)),
    torchvision.transforms.RandomHorizontalFlip(),      # 随机水平翻转
    torchvision.transforms.ToTensor(),                  # PIL图像 -> [0,1] 张量，并调整为 CHW
    # 按 CIFAR-10 经典统计量做逐通道标准化（均值/标准差），加速收敛
    torchvision.transforms.Normalize([0.4914, 0.4822, 0.4465],
                                     [0.2023, 0.1994, 0.2010])])

# 测试/验证集变换：不加随机性，只做确定性的张量化 + 标准化，保证评估可复现
transform_test = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.4914, 0.4822, 0.4465],
                                     [0.2023, 0.1994, 0.2010])])

In [ ]:
# ── 用 ImageFolder 读取整理好的图片，构建 4 个 Dataset ──────────────────
# ImageFolder 会自动把「子目录名」当作类别标签，并按字母序编号 0..9
# train_ds / train_valid_ds 用训练增广（带随机性）
train_ds, train_valid_ds = [
    torchvision.datasets.ImageFolder(
        os.path.join(data_dir, 'train_valid_test', folder),
        transform=transform_train) for folder in ['train', 'train_valid']]

# valid_ds / test_ds 用测试变换（确定性）
valid_ds, test_ds = [
    torchvision.datasets.ImageFolder(
        os.path.join(data_dir, 'train_valid_test', folder),
        transform=transform_test) for folder in ['valid', 'test']]

In [ ]:
# ── 构建 DataLoader（按批次读取数据的迭代器）──────────────────────────────
# 训练用迭代器：shuffle=True 打乱顺序；drop_last=True 丢弃凑不满一个 batch 的尾部
train_iter, train_valid_iter = [
    torch.utils.data.DataLoader(dataset, batch_size, shuffle=True, drop_last=True)
    for dataset in (train_ds, train_valid_ds)]

# 验证集：不打乱（shuffle=False），结果可复现
valid_iter = torch.utils.data.DataLoader(valid_ds, batch_size, shuffle=False, drop_last=True)
# 测试集：不打乱、且 drop_last=False（每张图都要预测，一张都不能丢）
test_iter = torch.utils.data.DataLoader(test_ds, batch_size, shuffle=False, drop_last=False)

In [ ]:
# ── 定义模型 ────────────────────────────────────────────────────────────
def get_net():
    num_classes = 10                       # CIFAR-10 共 10 个类别
    net = d2l.resnet18(num_classes, 3)     # 第二个参数 3 = 输入通道数（RGB 彩色图）
    return net

# 损失函数：交叉熵。reduction="none" 表示对每个样本单独返回 loss、不求和/平均，
# 方便后续在训练循环里手动累加（配合 d2l.train_batch_ch13 的统计方式）。
loss = nn.CrossEntropyLoss(reduction="none")

In [ ]:
# ── 训练函数 ────────────────────────────────────────────────────────────
# 【已修复 1】原形参写成单数 num_epoch，函数体却用复数 num_epochs，会触发 NameError。
#            此处统一为 num_epochs。
# 【已修复 2】原函数不返回 net，导致调用处的 net 仍是 CPU 上未包装的模型，
#            后续预测会报设备不匹配。此处在末尾 return net（已搬到 GPU 的版本）。
def train(net, train_iter, valid_iter, num_epochs, lr, wd, devices, lr_period, lr_decay):
    # SGD 优化器：带动量 momentum=0.9、权重衰减 wd（L2 正则）
    trainer = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    # 学习率调度器：每隔 lr_period 个 epoch，把学习率乘以 lr_decay
    scheduler = torch.optim.lr_scheduler.StepLR(trainer, lr_period, lr_decay)
    num_batches, timer = len(train_iter), d2l.Timer()
    legend = ['train loss', 'train acc']
    if valid_iter is not None:
        legend.append('valid acc')
    animator = d2l.Animator(xlabel='epoch', xlim=[1, num_epochs], legend=legend)
    # 多卡数据并行，并把模型搬到主 GPU；注意这是局部变量，故下文需 return 出去
    net = nn.DataParallel(net, device_ids=devices).to(devices[0])
    for epoch in range(num_epochs):
        net.train()                      # 切换到训练模式（启用 BN/Dropout 的训练行为）
        metric = d2l.Accumulator(3)      # 累加器：[loss之和, 正确数之和, 样本数之和]
        for i, (features, labels) in enumerate(train_iter):
            timer.start()
            # train_batch_ch13: 前向 + 反向 + 更新，返回该 batch 的 loss 和正确数
            l, acc = d2l.train_batch_ch13(net, features, labels, loss, trainer, devices)
            metric.add(l, acc, labels.shape[0])
            timer.stop()
            # 每个 epoch 内画 5 个点（以及最后一个 batch），实时观察曲线
            if (i + 1) % (num_batches // 5) == 0 or i == num_batches - 1:
                animator.add(epoch + (i + 1) / num_batches,
                             (metric[0] / metric[2], metric[1] / metric[2], None))
        if valid_iter is not None:       # 有验证集则每个 epoch 评估一次验证精度
            valid_acc = d2l.evaluate_accuracy_gpu(net, valid_iter)
            animator.add(epoch + 1, (None, None, valid_acc))
        scheduler.step()                 # 一个 epoch 结束，更新学习率
    # 打印最终指标与吞吐量（样本/秒）
    measures = (f'train loss {metric[0] / metric[2]:.3f},'
                f'train acc {metric[1] / metric[2]:.3f}')
    if valid_iter is not None:
        measures += f', valid acc {valid_acc:.3f}'
    print(measures + f'\n{metric[2] * num_epochs / timer.sum():.1f}'
          f' examples/sec on {str(devices)}')
    return net   # 返回已搬到 GPU 的模型，供后续预测使用

In [ ]:
# ── 训练并在验证集上评估模型 ────────────────────────────────────────────
# try_all_gpus: 返回所有可用 GPU 的设备列表（无 GPU 则返回 [cpu]）
devices, num_epochs, lr, wd = d2l.try_all_gpus(), 20, 2e-4, 5e-4
lr_period, lr_decay, net = 4, 0.9, get_net()   # 每隔 4 个 epoch，学习率 ×0.9
# 这里用 train_iter（90%）训练、valid_iter 评估，用于调超参；返回值可不接
train(net, train_iter, valid_iter, num_epochs, lr, wd, devices, lr_period, lr_decay)

In [ ]:
# ── 用全量数据重新训练，对测试集预测并生成提交文件 ──────────────────────
net, preds = get_net(), []
# 【已修复 2 落地处】用 net = train(...) 接收返回值：
#   train_valid_iter 用「训练集+验证集」的全部数据训练（valid_iter 传 None 不再评估），
#   返回的 net 已在 GPU 上且加载好权重，可直接用于预测。
net = train(net, train_valid_iter, None, num_epochs, lr, wd, devices, lr_period, lr_decay)

net.eval()   # 切换到评估模式（关闭 Dropout、固定 BN 统计量）
with torch.no_grad():                 # 预测阶段无需梯度，省显存、提速
    for X, _ in test_iter:            # 测试集标签是占位的 unknown，用 _ 忽略
        y_hat = net(X.to(devices[0]))                       # 前向得到 logits
        # argmax 取概率最大的类别索引；转 int32、搬回 CPU、转 numpy 收集
        preds.extend(y_hat.argmax(dim=1).type(torch.int32).cpu().numpy())

# Kaggle 要求 id 按「字符串字典序」排列：'1','10','100','2'...（不是数值序！）
sorted_ids = list(range(1, len(test_ds) + 1))
sorted_ids.sort(key=lambda x: str(x))
df = pd.DataFrame({'id': sorted_ids, 'label': preds})
# 把预测的类别索引(0~9) 映射回类名字符串（如 'frog'）；classes 顺序与训练时一致
df['label'] = df['label'].apply(lambda x: train_valid_ds.classes[x])
df.to_csv('submission.csv', index=False)   # 生成可提交到 Kaggle 的结果文件